# Controlled answer-only PEFT sensitivity experiment

This notebook tests one specific question: does masking the instruction and clause tokens, so that loss is calculated only on the category answer, change PEFT performance?

The official PEFT training configuration is retained: Mistral-7B-Instruct-v0.2, the frozen training and validation splits, the original prompt construction, LoRA rank 16, alpha 32, dropout 0.05, learning rate `2e-4`, effective batch size 16 and one epoch. The only substantive training-objective change is answer-only label masking.

Operational checkpointing does not alter the objective. Checkpoints are written to Google Drive every 100 steps, and an interrupted run resumes automatically. The test split is deliberately not loaded. Run through validation, send back the validation results, and decide on test evaluation only after the model and row-alignment audits pass.


## 1. Environment and persistent storage


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# Install the required packages. If Colab requests a restart after this cell,
# restart the session, then rerun this cell before continuing.
%pip install -q transformers accelerate bitsandbytes peft datasets scikit-learn


In [ ]:
import gc
import json
import os
import random
import re
from datetime import datetime, timezone

import accelerate
import bitsandbytes
import datasets
import numpy as np
import pandas as pd
import peft
import sklearn
import torch
import transformers
from datasets import load_dataset
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

DATA_PATH = "/content/drive/MyDrive/Thesis1_Project"
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"
SEED = 42
MAX_LENGTH = 2048

CHECKPOINT_DIR = os.path.join(DATA_PATH, "peft_answer_only_control_v4_checkpoints")
FINAL_ADAPTER_DIR = os.path.join(DATA_PATH, "peft_answer_only_control_v4_final_adapter")
VALIDATION_PARTIAL = os.path.join(DATA_PATH, "peft_answer_only_control_v4_validation_partial.csv")
VALIDATION_FINAL = os.path.join(DATA_PATH, "peft_answer_only_control_v4_validation_final.csv")
VALIDATION_METRICS = os.path.join(DATA_PATH, "peft_answer_only_control_v4_validation_metrics.json")
TRAINING_METADATA = os.path.join(DATA_PATH, "peft_answer_only_control_v4_training_metadata.json")
COMPLETE_MARKER = os.path.join(FINAL_ADAPTER_DIR, "training_complete.txt")

CHOICES = [
    "Parties", "License Grant", "Cap On Liability", "Audit Rights",
    "Anti-Assignment", "Insurance", "Document Name", "Agreement Date",
    "Governing Law", "Expiration Date", "Effective Date",
    "Post-Termination Services", "Revenue/Profit Sharing",
    "Minimum Commitment", "Exclusivity", "Rofr/Rofo/Rofn",
    "Ip Ownership Assignment", "Non-Transferable License", "Non-Compete",
    "Termination For Convenience",
]
CHOICES_STR = ", ".join(CHOICES)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before continuing.")
torch.cuda.manual_seed_all(SEED)

required_files = [
    os.path.join(DATA_PATH, "peft_train.jsonl"),
    os.path.join(DATA_PATH, "peft_val.jsonl"),
    os.path.join(DATA_PATH, "train.csv"),
    os.path.join(DATA_PATH, "val.csv"),
]
missing_files = [path for path in required_files if not os.path.exists(path)]
if missing_files:
    raise FileNotFoundError("Required frozen inputs are missing:\n" + "\n".join(missing_files))

train_df = pd.read_csv(os.path.join(DATA_PATH, "train.csv")).reset_index(drop=True)
val_df = pd.read_csv(os.path.join(DATA_PATH, "val.csv")).reset_index(drop=True)
if len(train_df) != 8257 or len(val_df) != 833:
    raise ValueError("The frozen training or validation split has changed.")
for split_name, frame in [("training", train_df), ("validation", val_df)]:
    if set(frame["clause_type"].unique()) != set(CHOICES):
        raise ValueError(f"{split_name.title()} labels do not match the frozen 20 classes.")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Training clauses: {len(train_df):,}")
print(f"Validation clauses: {len(val_df):,}")
print(f"Transformers: {transformers.__version__}")
print(f"PEFT: {peft.__version__}")
print(f"Datasets: {datasets.__version__}")
print(f"Accelerate: {accelerate.__version__}")


## 2. Original prompt with answer-only labels

The source JSONL files and prompt wording are the same as in the official PEFT run. Prompt tokens receive label `-100`; only the category answer and end-of-sequence token contribute to the causal-language-model loss.


In [ ]:
def build_prompt(instruction):
    messages = [{"role": "user", "content": instruction}]
    if getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    return f"<s>[INST] {instruction} [/INST]"


def build_official_instruction(example):
    return (
        f"{example['instruction']}\n\n"
        f"Clause:\n{example['input']}\n\n"
        "Return ONLY the category name. Do not explain your answer and do not add punctuation."
    )


def tokenize_answer_only(example):
    prompt = build_prompt(build_official_instruction(example))
    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    answer_ids = tokenizer(
        " " + str(example["output"]).strip() + tokenizer.eos_token,
        add_special_tokens=False,
    )["input_ids"]

    available_prompt_tokens = MAX_LENGTH - len(answer_ids)
    if available_prompt_tokens <= 0:
        raise ValueError("A category answer exceeds MAX_LENGTH.")

    # Match the official run's right-side truncation at the complete-sequence limit.
    prompt_ids = prompt_ids[:available_prompt_tokens]
    input_ids = prompt_ids + answer_ids
    labels = [-100] * len(prompt_ids) + answer_ids.copy()

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels,
        "prompt_token_count": len(prompt_ids),
        "answer_token_count": len(answer_ids),
    }


raw_train = load_dataset(
    "json",
    data_files=os.path.join(DATA_PATH, "peft_train.jsonl"),
    split="train",
)
raw_validation = load_dataset(
    "json",
    data_files=os.path.join(DATA_PATH, "peft_val.jsonl"),
    split="train",
)

required_columns = {"instruction", "input", "output"}
for split_name, dataset in [("training", raw_train), ("validation", raw_validation)]:
    missing = required_columns - set(dataset.column_names)
    if missing:
        raise ValueError(f"{split_name.title()} data is missing: {sorted(missing)}")

if len(raw_train) != 8257 or len(raw_validation) != 833:
    raise ValueError("The PEFT JSONL row counts no longer match the frozen splits.")
if set(raw_train["output"]) != set(CHOICES) or set(raw_validation["output"]) != set(CHOICES):
    raise ValueError("The PEFT JSONL labels do not match the frozen schema.")

train_dataset = raw_train.map(
    tokenize_answer_only,
    remove_columns=raw_train.column_names,
    desc="Tokenizing answer-only training examples",
)
validation_loss_dataset = raw_validation.map(
    tokenize_answer_only,
    remove_columns=raw_validation.column_names,
    desc="Tokenizing answer-only validation examples",
)

for split_name, dataset in [("training", train_dataset), ("validation", validation_loss_dataset)]:
    if min(dataset["answer_token_count"]) <= 0:
        raise RuntimeError(f"{split_name.title()} contains a fully masked example.")
    for row in dataset.select(range(min(200, len(dataset)))):
        first_supervised = next(i for i, value in enumerate(row["labels"]) if value != -100)
        if first_supervised != row["prompt_token_count"]:
            raise RuntimeError(f"{split_name.title()} masking boundary is inconsistent.")
        if any(value != -100 for value in row["labels"][:first_supervised]):
            raise RuntimeError(f"{split_name.title()} prompt masking failed.")
        if row["labels"][first_supervised:] != row["input_ids"][first_supervised:]:
            raise RuntimeError(f"{split_name.title()} answer labels do not match input tokens.")

sample = train_dataset[0]
supervised_ids = [value for value in sample["labels"] if value != -100]
print("Decoded supervised tokens:", tokenizer.decode(supervised_ids, skip_special_tokens=False))
print("Prompt tokens:", sample["prompt_token_count"])
print("Supervised answer tokens:", sample["answer_token_count"])
print("Masking audit passed: prompt tokens are ignored and answer tokens are supervised.")


## 3. Controlled one-epoch QLoRA training

The hyperparameters match the official PEFT run. Checkpoints are stored directly in Drive every 100 steps. If Colab disconnects, rerun the notebook through this cell; it will resume from the latest v4 checkpoint. Do not point this notebook at the v3 or official checkpoint directories.


In [ ]:
from dataclasses import dataclass
from transformers import Trainer, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint


@dataclass
class AnswerOnlyCollator:
    pad_token_id: int

    def __call__(self, features):
        max_length = max(len(item["input_ids"]) for item in features)
        input_ids, attention_masks, labels = [], [], []
        for item in features:
            padding = max_length - len(item["input_ids"])
            input_ids.append(item["input_ids"] + [self.pad_token_id] * padding)
            attention_masks.append(item["attention_mask"] + [0] * padding)
            labels.append(item["labels"] + [-100] * padding)
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_masks, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


if os.path.exists(COMPLETE_MARKER):
    print("Training-complete marker found. Training will not run again.")
    print("Final adapter:", FINAL_ADAPTER_DIR)
else:
    for variable_name in ["model", "trainer", "base_model"]:
        if variable_name in globals():
            del globals()[variable_name]
    gc.collect()
    torch.cuda.empty_cache()

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        low_cpu_mem_usage=True,
        dtype=torch.float16,
    )
    model = prepare_model_for_kbit_training(
        model,
        use_gradient_checkpointing=True,
    )
    model.config.use_cache = False

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    for parameter in model.parameters():
        if parameter.requires_grad:
            parameter.data = parameter.data.to(torch.float32)

    training_args = TrainingArguments(
        output_dir=CHECKPOINT_DIR,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=16,
        learning_rate=2e-4,
        num_train_epochs=1,
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=3,
        logging_steps=10,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        fp16=True,
        bf16=False,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        optim="paged_adamw_8bit",
        report_to=[],
        remove_unused_columns=False,
        seed=SEED,
        data_seed=SEED,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=validation_loss_dataset,
        data_collator=AnswerOnlyCollator(tokenizer.pad_token_id),
    )

    model.print_trainable_parameters()
    trainable_dtypes = {
        parameter.dtype
        for parameter in model.parameters()
        if parameter.requires_grad
    }
    if trainable_dtypes != {torch.float32}:
        raise RuntimeError(f"Unexpected trainable parameter dtypes: {trainable_dtypes}")

    last_checkpoint = (
        get_last_checkpoint(CHECKPOINT_DIR)
        if os.path.isdir(CHECKPOINT_DIR)
        else None
    )
    if last_checkpoint:
        print("Resuming controlled v4 run from:", last_checkpoint)
    else:
        print("Starting controlled v4 run from step zero.")

    train_result = trainer.train(resume_from_checkpoint=last_checkpoint)
    trainer.save_model(FINAL_ADAPTER_DIR)
    tokenizer.save_pretrained(FINAL_ADAPTER_DIR)
    trainer.state.save_to_json(os.path.join(FINAL_ADAPTER_DIR, "trainer_state.json"))
    os.makedirs(FINAL_ADAPTER_DIR, exist_ok=True)
    with open(COMPLETE_MARKER, "w", encoding="utf-8") as file:
        file.write("complete\n")

    metadata = {
        "study": "Controlled answer-only PEFT sensitivity V4",
        "relationship_to_primary": "Same primary hyperparameters; prompt tokens masked from loss",
        "test_loaded": False,
        "model_id": MODEL_ID,
        "seed": SEED,
        "epochs": 1,
        "learning_rate": 2e-4,
        "effective_batch_size": 16,
        "max_length": MAX_LENGTH,
        "answer_only_loss": True,
        "best_validation_loss_checkpoint": trainer.state.best_model_checkpoint,
        "best_validation_loss": trainer.state.best_metric,
        "global_step": trainer.state.global_step,
        "transformers_version": transformers.__version__,
        "peft_version": peft.__version__,
        "datasets_version": datasets.__version__,
        "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    with open(TRAINING_METADATA, "w", encoding="utf-8") as file:
        json.dump(metadata, file, indent=2)

    print("Training completed at step:", trainer.state.global_step)
    print("Best validation-loss checkpoint:", trainer.state.best_model_checkpoint)
    print("Final selected adapter:", FINAL_ADAPTER_DIR)


## 4. Validation generation

This section evaluates the saved, validation-loss-selected adapter on all 833 validation clauses. Predictions are checkpointed to Drive every batch. The test split is still not loaded.


In [ ]:
def parse_prediction(raw_output):
    if raw_output is None or pd.isna(raw_output):
        return "Unknown"
    text = str(raw_output).strip()
    if not text:
        return "Unknown"
    candidates = [text, text.splitlines()[0].strip()]
    for candidate in candidates:
        candidate = re.sub(r"^[\s`*_\"'“”‘’]+|[\s`*_\"'“”‘’]+$", "", candidate).strip()
        category_match = re.fullmatch(r"(?i)category\s*:\s*(.+)", candidate)
        if category_match:
            candidate = category_match.group(1).strip()
        candidate = re.sub(r"[.,;:!?]+$", "", candidate).strip()
        for choice in CHOICES:
            if candidate.casefold() == choice.casefold():
                return choice
    return "Unknown"


if not os.path.exists(os.path.join(FINAL_ADAPTER_DIR, "adapter_config.json")):
    raise FileNotFoundError("The final v4 adapter is missing. Complete training first.")

for variable_name in ["model", "trainer", "base_model", "evaluation_model"]:
    if variable_name in globals():
        del globals()[variable_name]
gc.collect()
torch.cuda.empty_cache()

if os.path.exists(VALIDATION_FINAL):
    validation_results = pd.read_csv(VALIDATION_FINAL)
    print("Complete validation predictions already exist.")
elif os.path.exists(VALIDATION_PARTIAL):
    validation_results = pd.read_csv(VALIDATION_PARTIAL)
    validation_results = validation_results.drop_duplicates("row_id", keep="last")
    print(f"Resuming validation at {len(validation_results):,}/{len(val_df):,} rows.")
else:
    validation_results = pd.DataFrame(
        columns=["row_id", "clause_text", "clause_type", "predicted_label", "raw_output"]
    )

validation_results["row_id"] = pd.to_numeric(
    validation_results["row_id"], errors="coerce"
).astype("Int64")
completed_ids = set(validation_results["row_id"].dropna().astype(int))
pending = val_df.loc[~val_df.index.isin(completed_ids)].copy()

if len(pending):
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        low_cpu_mem_usage=True,
        dtype=torch.float16,
    )
    evaluation_model = PeftModel.from_pretrained(
        base_model,
        FINAL_ADAPTER_DIR,
        is_trainable=False,
    )
    evaluation_model.eval()
    evaluation_model.config.use_cache = True
    tokenizer.padding_side = "left"

    batch_size = 4
    for start in range(0, len(pending), batch_size):
        batch = pending.iloc[start:start + batch_size]
        prompts = [
            build_prompt(
                "Classify the legal clause below into exactly ONE of these categories:\n"
                f"{CHOICES_STR}\n\n"
                "Return ONLY the category name. Do not explain your answer and do not add punctuation.\n\n"
                f"Clause:\n{text}"
            )
            for text in batch["clause_text"]
        ]
        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=4096,
        ).to("cuda")
        with torch.inference_mode():
            generated = evaluation_model.generate(
                **inputs,
                max_new_tokens=12,
                do_sample=False,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id,
            )
        generated_tokens = generated[:, inputs["input_ids"].shape[1]:]
        raw_outputs = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
        rows = []
        for row_id, (_, row), raw_output in zip(batch.index, batch.iterrows(), raw_outputs):
            rows.append({
                "row_id": int(row_id),
                "clause_text": row["clause_text"],
                "clause_type": row["clause_type"],
                "predicted_label": parse_prediction(raw_output),
                "raw_output": raw_output.strip(),
            })
        validation_results = pd.concat(
            [validation_results, pd.DataFrame(rows)],
            ignore_index=True,
        )
        validation_results["row_id"] = validation_results["row_id"].astype(int)
        validation_results = (
            validation_results
            .drop_duplicates("row_id", keep="last")
            .sort_values("row_id")
        )
        validation_results.to_csv(VALIDATION_PARTIAL, index=False)
        if len(validation_results) % 100 < batch_size or len(validation_results) == len(val_df):
            print(f"Validation: {len(validation_results):,}/{len(val_df):,}")

    del evaluation_model, base_model
    gc.collect()
    torch.cuda.empty_cache()

validation_results["row_id"] = validation_results["row_id"].astype(int)
validation_results = validation_results.drop_duplicates("row_id", keep="last").sort_values("row_id")
if len(validation_results) != len(val_df):
    raise RuntimeError("Validation predictions are incomplete.")
if set(validation_results["row_id"]) != set(range(len(val_df))):
    raise RuntimeError("Validation row IDs are misaligned.")
if not validation_results["clause_type"].reset_index(drop=True).equals(
    val_df["clause_type"].reset_index(drop=True)
):
    raise RuntimeError("Validation reference labels are misaligned.")

validation_results.to_csv(VALIDATION_FINAL, index=False)
y_true = validation_results["clause_type"]
y_pred = validation_results["predicted_label"]
metrics = {
    "accuracy": accuracy_score(y_true, y_pred),
    "macro_precision": precision_score(y_true, y_pred, labels=CHOICES, average="macro", zero_division=0),
    "macro_recall": recall_score(y_true, y_pred, labels=CHOICES, average="macro", zero_division=0),
    "macro_f1": f1_score(y_true, y_pred, labels=CHOICES, average="macro", zero_division=0),
    "weighted_f1": f1_score(y_true, y_pred, labels=CHOICES, average="weighted", zero_division=0),
    "unknown_count": int(y_pred.eq("Unknown").sum()),
    "unknown_rate": float(y_pred.eq("Unknown").mean()),
    "rows": int(len(validation_results)),
    "test_evaluated": False,
}
with open(VALIDATION_METRICS, "w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=2)

display(pd.DataFrame([metrics]).round(4))
print("\nValidation classification report:\n")
print(classification_report(y_true, y_pred, labels=CHOICES, zero_division=0))
print("Validation evaluation complete.")
